# 01 — Sensitivity analysis

Run from repo root or keep `sys.path` as below.


In [ ]:
import sys
sys.path.insert(0, "..")
import torch
from autoquant import AutoQuantizer

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")


In [ ]:
quantizer = AutoQuantizer("gpt2")
sensitivity = quantizer.analyze_sensitivity(num_samples=50)


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.DataFrame(
    [
        {
            "layer": name.split(".")[-2] + "." + name.split(".")[-1],
            "full_name": name,
            "score": score,
        }
        for name, score in sensitivity.items()
    ]
).sort_values("score", ascending=False)

plt.figure(figsize=(14, 6))
colors = [
    "#d62728" if s > 0.7 else "#ff7f0e" if s > 0.4 else "#1f77b4"
    for s in df["score"]
]
plt.bar(range(len(df)), df["score"], color=colors)
plt.xlabel("Layer (sorted by sensitivity)")
plt.ylabel("Sensitivity Score")
plt.title("Per-layer sensitivity (GPT-2)")
plt.xticks([])
plt.tight_layout()
plt.savefig("sensitivity_plot.png", dpi=150)
plt.show()


In [ ]:
print("Top 10 most sensitive:")
for _, row in df.head(10).iterrows():
    print(f"  {row['full_name'][:60]:60s} {row['score']:.3f}")
